# Preprocessing and Baseline Classification

## Objective

This notebook prepares the gene-expression dataset for machine-learning classification and establishes a baseline against which more complex models can be compared.

The aim is to investigate whether gene-expression profiles can distinguish healthy controls, bipolar disorder and schizophrenia samples.

## Planned workflow

1. Load the expression data and sample metadata.
2. Exclude the sample flagged during exploratory data analysis.
3. Check data alignment and prepare the target labels.
4. Create a reproducible, stratified train/test split.
5. Handle missing expression values without leaking information from the test set.
6. Evaluate a dummy baseline classifier.

## Import packages, set up paths

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

RANDOM_STATE = 42

PROJECT_DIR = Path.cwd().parent
DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"

print("Project directory:", PROJECT_DIR)
print("Processed data directory:", PROCESSED_DIR)
print("Imports successful.")

Project directory: C:\Users\taraj\OneDrive\Documents\GitHub\psychiatric-gene-expression-ml
Processed data directory: C:\Users\taraj\OneDrive\Documents\GitHub\psychiatric-gene-expression-ml\data\processed
Imports successful.


## Check data

In [9]:
X = pd.read_csv(PROCESSED_DIR / "expression_matrix.csv", index_col=0)

metadata_clean = pd.read_csv(PROCESSED_DIR / "sample_metadata.csv")

X.index = X.index.astype(str)
metadata_clean["sample_id"] = metadata_clean["sample_id"].astype(str)

print("Expression matrix shape:", X.shape)
print("Metadata shape:", metadata_clean.shape)
print("Expression sample IDs unique:", X.index.is_unique)
print("Metadata sample IDs unique:", metadata_clean["sample_id"].is_unique)

Expression matrix shape: (88, 30635)
Metadata shape: (88, 9)
Expression sample IDs unique: True
Metadata sample IDs unique: True


## Exclude GSM1521674

In [4]:
# Sample flagged during exploratory data analysis.
EXCLUDED_SAMPLE = "GSM1521674"

# Confirm that the sample is present before excluding it.
assert EXCLUDED_SAMPLE in X.index, "Sample not found in expression matrix."

metadata_ids = set(metadata_clean["sample_id"])
assert EXCLUDED_SAMPLE in metadata_ids, "Sample not found in metadata."

# Exclude the sample from both datasets.
X = X.drop(index=EXCLUDED_SAMPLE)

metadata_clean = metadata_clean[
    metadata_clean["sample_id"] != EXCLUDED_SAMPLE
].copy()

# Check sample counts and alignment.
assert len(X) == 87
assert len(metadata_clean) == 87
assert set(X.index) == set(metadata_clean["sample_id"])

print("Excluded sample:", EXCLUDED_SAMPLE)
print("Remaining expression matrix:", X.shape)
print("Remaining metadata:", metadata_clean.shape)
print("\nRemaining class distribution:")
print(metadata_clean["disease state"].value_counts())

Excluded sample: GSM1521674
Remaining expression matrix: (87, 30635)
Remaining metadata: (87, 9)

Remaining class distribution:
disease state
bipolar disorder    29
healthy control     29
schizophrenia       29
Name: count, dtype: int64


## Prep target labels

In [5]:
# Create the target labels in the same order as the expression matrix.
metadata_indexed = metadata_clean.set_index("sample_id")
y = metadata_indexed.loc[X.index, "disease state"]

# Confirm that the features and labels are aligned.
assert X.index.equals(y.index)
assert y.notna().all()

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("\nTarget class distribution:")
print(y.value_counts())
print("\nTarget labels:")
print(sorted(y.unique()))

Feature matrix shape: (87, 30635)
Target shape: (87,)

Target class distribution:
disease state
bipolar disorder    29
healthy control     29
schizophrenia       29
Name: count, dtype: int64

Target labels:
['bipolar disorder', 'healthy control', 'schizophrenia']


## Split training and test data

In [6]:
# Split the data before imputing missing values or fitting models.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Training features:", X_train.shape)
print("Test features:", X_test.shape)

print("\nTraining class distribution:")
print(y_train.value_counts())

print("\nTest class distribution:")
print(y_test.value_counts())

# Confirm that no sample appears in both sets.
assert set(X_train.index).isdisjoint(set(X_test.index))
assert X_train.index.equals(y_train.index)
assert X_test.index.equals(y_test.index)

print("\nTrain/test checks passed.")

Training features: (69, 30635)
Test features: (18, 30635)

Training class distribution:
disease state
bipolar disorder    23
schizophrenia       23
healthy control     23
Name: count, dtype: int64

Test class distribution:
disease state
schizophrenia       6
healthy control     6
bipolar disorder    6
Name: count, dtype: int64

Train/test checks passed.


## Build Baseline model pipeline

In [7]:
# Create a pipeline that imputes missing values using training-set medians.
baseline_model = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", DummyClassifier(strategy="most_frequent")),
    ]
)

print("Baseline pipeline created.")
print(baseline_model)

Baseline pipeline created.
Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('classifier', DummyClassifier(strategy='most_frequent'))])


## Fit and run model

In [8]:
# Fit the pipeline using training data only.
baseline_model.fit(X_train, y_train)

# Predict diagnoses for the held-out test set.
y_pred_baseline = baseline_model.predict(X_test)

# Calculate baseline metrics.
baseline_accuracy = accuracy_score(y_test, y_pred_baseline)

print(f"Baseline accuracy: {baseline_accuracy:.3f}")

print("\nClassification report:")
print(
    classification_report(
        y_test,
        y_pred_baseline,
        labels=sorted(y.unique()),
        zero_division=0,
    )
)

print("\nConfusion matrix:")
print(confusion_matrix(
        y_test,
        y_pred_baseline,
        labels=sorted(y.unique()),
    ))

Baseline accuracy: 0.333

Classification report:
                  precision    recall  f1-score   support

bipolar disorder       0.33      1.00      0.50         6
 healthy control       0.00      0.00      0.00         6
   schizophrenia       0.00      0.00      0.00         6

        accuracy                           0.33        18
       macro avg       0.11      0.33      0.17        18
    weighted avg       0.11      0.33      0.17        18


Confusion matrix:
[[6 0 0]
 [6 0 0]
 [6 0 0]]


## Baseline Results

A `DummyClassifier` using the most-frequent strategy was evaluated on the held-out test set after median imputation.

The model achieved an accuracy of 33.3% and a macro F1-score of 0.17. It predicted bipolar disorder for all 18 test samples, correctly classifying the six bipolar disorder samples but failing to identify healthy controls or schizophrenia samples.

These results establish a baseline for comparison with models that learn from gene-expression features. The test set contains only 18 samples, so performance estimates are uncertain.